In [ ]:
## import libraries ##
from pathlib import Path

## import custom functions
from utility_functions import *
from processing_functions import *

In [ ]:
# settings for processing pipeline are set in processing_parameters.py!

# user input
# select spike sorter and files for processing
spike_sorter = "kilosort4" # currently implemented: "kilosort4"
recordings_folder = Path("S:/Fakultaet/MFZ/NWFZ/AG-deHoz-Scratch/Neuropixels_DO_NOT_DELETE") # path to all recordings, use Path()
recordings_to_process = ["F02_03_2026_2416_g0", "F01_03_2026_2413_g0"] # list of names of recordings to process

local_output_folder = Path("") # local folder to temporarily save output for faster processing, preferably SSD, use Path()
final_output_folder = Path("S:/Fakultaet/MFZ/NWFZ/AG-deHoz-Scratch/Fabian_scratch/Neuropixels_processed") # parent folder where output is saved, use Path()


## initialize current data to process
neuropixels_data = NeuropixelsData(recordings_folder, recordings_to_process, spike_sorter, local_output_folder, final_output_folder)

In [ ]:
### pre-processing ###
from processing_parameters import preprocessing_configurations

neuropixels_data.run_preprocessing(preprocessing_configurations, ignore_existing_files = False)

Folder structure exists as expected.

Chosen spike sorter: kilosort4

------------------preprocessing parameters------------------
{'spike_interface': {'phase_shift': {},
                     'bandpass_filter': {'freq_min': 300, 'freq_max': 6000}},
 'custom_steps': {'correct_bad_channels': {'method': 'coherence+psd'},
                  'si_preprocessing_after_channel_removal': {}}}

------------------------------------------------------------

Processing recording F02_03_2026_2416_g0...
2 probe(s) found
Processing probe 1/2...
SpikeGLXRecordingExtractor: 384 channels - 29999.993855 Hz - 1 segments - 127,321,164 samples 
                            4,244.04s (1.18 hours) - int16 dtype - 91.07 GiB


KeyboardInterrupt: 

In [ ]:
### spike sorting ###
from processing_parameters import spike_sorting_configurations

neuropixels_data.run_spike_sorting(spike_sorting_configurations)

In [ ]:
### post processing ###
from processing_parameters import postprocessing_configurations

neuropixels_data.run_postprocessing(postprocessing_configurations, delete_preprocessing_data = False, delete_raw_spike_sorting = False, overwrite_existing_files = False)

In [ ]:
### curation ###
from processing_parameters import curation_thresholds

neuropixels_data.automatic_curation("bombcell", curation_thresholds)

In [ ]:
## questions
# quality metrics on recording level in beginning?

# does output folder have to exist or does spike interface handle this?

# keep raw spike sorting data (currently in temporary) or not?

# beneficial to add some plots?
# e.g.: # bombcell plots (see https://spikeinterface.readthedocs.io/en/latest/how_to/auto_label_units.html)

# is curated sorting analyzer saved as new file or overwritten?

In [ ]:
# figures -> which ones do you need?


## visualization

# -> widget module is simplest way
# widget object -> access to underlying matplotlib fig & ax

# spikeinterface-gui -> spike sorting output

si_widgets.plot_sorting_summary(analyzer, curation=True)
si_widgets.plot_quality_metrics(analyzer)

# can also export locally to Phy and use phy for manula curation -> phy needs special env cause some other dependencies, si_export.export_to_phy(analyzer, "folder", verbose = True)
# then after curation in phy yopu can load via curated_sorting = si_extractors.read_phy("folder", exclude_cluster_groups=["noise"])


si.export_report(analyzer_clean, base_folder / 'report', format='png') # export figures
# load via analyzer_clean = si.load_sorting_analyzer(base_folder / 'analyzer_clean')


## plot basic overview
traces_basic_plot = si_widgets.plot_traces(recording, time_range=(0,5))

channel_ids = recording.get_channel_ids()
recording_fs = recording.get_sampling_frequency()
n_channels = recording.get_num_channels()
n_segments = recording.get_num_segments()

print("Channel ids:", channel_ids)
print("Sampling frequency of recording:", recording_fs)
print("Number of channels:", n_channels)
print("Number of segments:", n_segments)

probe = recording.get_probe() # if not saved in recording you can set this using set_probe


## visualize pre-procesing
%matplotlib widget
si_widgets.plot_traces({'filter':rec1, 'cmr': rec4}, backend='ipywidgets')

# or 

fig, axs = plt.subplots(ncols=3, figsize=(20, 10))

si.plot_traces(rec1, backend='matplotlib',  clim=(-50, 50), ax=axs[0])
si.plot_traces(rec4, backend='matplotlib',  clim=(-50, 50), ax=axs[1])
si.plot_traces(rec, backend='matplotlib',  clim=(-50, 50), ax=axs[2])
for i, label in enumerate(('filter', 'cmr', 'final')):
    axs[i].set_title(label)



# check spiking activity & drift before spike sorting
# probably not necessary if you use kilosort, as it does drift correction
noise_levels_uV = si.get_noise_levels(recording_preprocessed, return_in_uV=True)
fig, ax = plt.subplots()
_ = ax.hist(noise_levels_uV, bins=np.arange(5, 30, 2.5))
ax.set_xlabel("noise (uV)")
ax.set_ylabel("counts")


## GUI
GUI = si_widgets.plot_sorting_summary(analyzer, backend = "spikeinterface_gui")
# reads in sorting analyzer objects, produces curation files



In [ ]:
## todo

continue with https://spikeinterface.readthedocs.io/en/latest/how_to/auto_curation_training.html

# decide on parameters to use

# do preprocessing (replicate catgt?)

# do spike sorting

# try with good SNR AC recording, if it looks good process rest

# mulit-shank recordings -> process by channel group!, implement option (https://spikeinterface.readthedocs.io/en/latest/how_to/process_by_channel_group.html)
# also if recording from different areas along shank might be useful to sort different areas with differently/with different sorters
# recording.set_property("group", grouping_array)
# split_recording = recording.split_by("group") # dictionary containing seperate recordings
# feeding split recordings into preprocessing functions -> pre-processed separately, returns dict again
# can also sort by channel group -> pass split recording to run_sorter function, returns dict, can then be passed to sorting analyzer


# read https://spikeinterface.readthedocs.io/en/latest/modules/index.html or add to todo

# use hybrid recordings to bench-mark parameters for different areas?

SyntaxError: invalid syntax (2188548053.py, line 7)

In [12]:
# compression? -> save storage space
# fropm wavpack_numcodecs imprt WavPack
# define compressor = WavPack(level=3)
# save compressed data recording = recording.save(format="zarr", folder="xyz", compressor = compressor, n_jobs=16?)

# nwb files to share data